Runnable implementation of the REINFORCE algorithm from the [policy-gradient lecture](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce). The agent learns a stochastic policy $\pi_\theta(a \mid s)$ over the two CartPole actions, using Monte Carlo returns $G_t$ as an unbiased estimator of the action-value $Q^\pi(s, a)$ under the current policy inside the policy gradient

$$
\nabla_\theta J(\pi_\theta) = \mathbb{E}_\pi\left[\nabla_\theta \log \pi_\theta(a \mid s)\, G_t\right].
$$

CartPole-v1 is considered solved at an average episode return of $475$ (max is $500$).

## The CartPole-v1 problem

CartPole is a classical control benchmark introduced by Barto, Sutton, and Anderson (1983) and now the "hello world" of deep RL. A pole is hinged on a cart that slides along a frictionless track. The agent chooses, at every timestep, whether to push the cart **left** or **right**; gravity and the cart's motion together determine how the pole tilts. The agent's task is to keep the pole upright for as long as possible.

![Gymnasium CartPole-v1 render — a black cart on a horizontal track with a light-brown pole standing straight up](images/cartpole.png)

*The Gymnasium `CartPole-v1` render: black cart, vertical pole, horizontal track. The environment emits one such frame per step.*

### Observation, action, reward

| | | |
|---|---|---|
| **Observation** (state $s$) | 4-D continuous | $[x,\, \dot x,\, \theta,\, \dot\theta]$ — cart position, cart velocity, pole angle (radians), pole angular velocity |
| **Action** (action $a$) | 2 discrete | $0$ = push left, $1$ = push right (no "do nothing" action) |
| **Reward** | $+1$ | granted for every timestep the pole stays up (including the terminal step) |
| **Episode length** | $\leq 500$ | hard cap in `v1` (was $200$ in `v0`) |

### Termination

An episode ends as soon as any of the following occurs:

- **pole fell too far** — $\lvert\theta\rvert > 12°$ ($\approx 0.2095$ rad)
- **cart left the track** — $\lvert x \rvert > 2.4$
- **time-limit reached** — $t = 500$ (truncation, not failure)

Because each step grants reward $1$, the **return** for an episode equals its length. The best possible return is $500$.

### "Solved" threshold

Gymnasium considers CartPole-v1 solved when the agent averages **$\geq 475$ return over $100$ consecutive episodes**. Our training loop below reports a simpler per-episode check (`total_reward > 475`) as a quick indicator of progress — the true solved criterion is a sliding average.

### Why it's a good benchmark for REINFORCE

The state space is small and continuous, the action set is tiny, and the reward signal is dense (every step contributes), so a stochastic policy can learn from just a few hundred episodes without a value baseline. This makes CartPole useful for *seeing* a policy-gradient signal at all — and also for exposing its limitations: the variance and stability issues discussed on the [parent lecture page](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce#why-the-learning-curve-is-not-monotonic) show up clearly here, even though the environment itself is "easy."

In [ ]:
%matplotlib inline

from __future__ import annotations
from pathlib import Path
import logging
import os

# Silence wandb's verbose init banner so only a single one-line summary
# reaches the rendered MDX. Must be set *before* `import wandb`.
os.environ.setdefault("WANDB_SILENT", "true")
os.environ.setdefault("WANDB_CONSOLE", "off")

from torch.distributions import Categorical
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

gamma = 0.99
num_episodes = 300
learning_rate = 0.01
hidden_dim = 64


_NB_NAME = "reinforce-cartpole.ipynb"


def _notebook_dir() -> Path:
    pm_input = os.environ.get("PM_INPUT_PATH")
    if pm_input and Path(pm_input).is_file():
        return Path(pm_input).resolve().parent
    for candidate in Path.cwd().rglob(_NB_NAME):
        return candidate.resolve().parent
    return Path.cwd()


IMAGES_DIR = _notebook_dir() / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


# --- Weights & Biases experiment tracking ------------------------------------
# All runs land in the shared `pantelis/eng-ai-agents` project so training
# curves and the harness's system telemetry are observable side-by-side (same
# project the `notebooks/scripts/wandb_utils.py` harness uses). When the
# harness has already started a run in the parent process and exported
# `WANDB_RUN_ID` to the kernel, wandb will *attach* to that run via
# `resume="allow"`; otherwise a fresh run named `reinforce-cartpole` is
# created inside the same project.
#
# Override the project/entity with WANDB_PROJECT / WANDB_ENTITY, or set
# WANDB_MODE=disabled to run fully offline.
_use_wandb = False
try:
    logging.getLogger("wandb").setLevel(logging.WARNING)
    import wandb

    if wandb.run is None:
        wandb.init(
            project=os.environ.get("WANDB_PROJECT", "eng-ai-agents"),
            entity=os.environ.get("WANDB_ENTITY", "pantelis"),
            name="reinforce-cartpole",
            group="reinforcement-learning",
            tags=["reinforcement-learning", "policy-gradient", "cartpole"],
            config={
                "algorithm": "REINFORCE",
                "environment": "CartPole-v1",
                "gamma": gamma,
                "num_episodes": num_episodes,
                "learning_rate": learning_rate,
                "hidden_dim": hidden_dim,
            },
            mode=os.environ.get("WANDB_MODE", "online"),
            resume="allow",
            reinit="finish_previous",
        )
    # Declare episode as the x-axis for every training metric so wandb's
    # auto-generated line charts plot against episode, not an internal step.
    wandb.define_metric("episode")
    for metric in ("loss", "return", "return_mean_50"):
        wandb.define_metric(metric, step_metric="episode")
    _use_wandb = wandb.run.settings.mode != "disabled"
    if _use_wandb:
        print(f"wandb dashboard: {wandb.run.url}")
    else:
        print("wandb running in disabled mode (no credentials) — training without metric streaming")
except Exception as exc:
    print(f"wandb init failed ({exc}); training will run without metric streaming")


def save_svg(fig, name: str) -> Path:
    """Save `fig` to images/<name>.svg and close it.

    Closing suppresses any inline image output so the generated MDX does not
    receive an auto-inserted <Frame><img/></Frame> block from the converter.
    Zoomable SVGs are referenced explicitly from the following markdown cell.
    """
    path = IMAGES_DIR / f"{name}.svg"
    fig.savefig(path, format="svg", bbox_inches="tight")
    plt.close(fig)
    return path

## Policy network and REINFORCE update

The policy is a small MLP with a single hidden layer of 64 units. `act` samples an action from the categorical distribution over the network's logits and stores `log π(a|s)` for the later gradient-ascent update. `train` computes the discounted return at each step (backward sweep for efficiency), forms the REINFORCE loss $-\sum_t \log \pi_\theta(a_t \mid s_t)\, G_t$, and performs one optimizer step per episode.

In [ ]:
class Pi(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_dim),
        )
        self.onpolicy_reset()
        self.train()

    def onpolicy_reset(self):
        self.log_probs = []
        self.rewards = []

    def forward(self, x):
        return self.model(x)

    def act(self, state):
        x = torch.as_tensor(state, dtype=torch.float32)
        logits = self.forward(x)
        pd = Categorical(logits=logits)
        action = pd.sample()
        self.log_probs.append(pd.log_prob(action))
        return action.item()


def train_episode(pi, optimizer):
    T = len(pi.rewards)
    rets = np.empty(T, dtype=np.float32)
    future_ret = 0.0
    for t in reversed(range(T)):
        future_ret = pi.rewards[t] + gamma * future_ret
        rets[t] = future_ret
    rets = torch.as_tensor(rets, dtype=torch.float32)
    log_probs = torch.stack(pi.log_probs)
    loss = -(log_probs * rets).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

## Training loop

CartPole is a very forgiving environment; a fresh, untuned REINFORCE agent typically reaches the 475-return solved threshold within a few hundred episodes. Per-episode **loss**, **return**, and a running **50-episode mean return** are streamed to Weights & Biases (free at [wandb.ai](https://wandb.ai/)) rather than printed to the notebook — this keeps the rendered page short and the metrics interactive. If wandb is not configured, training still runs but metrics are only summarized at the end.

In [ ]:
env = gym.make("CartPole-v1")
in_dim = env.observation_space.shape[0]  # 4
out_dim = env.action_space.n              # 2
pi = Pi(in_dim, out_dim)
optimizer = optim.Adam(pi.parameters(), lr=learning_rate)

episode_returns: list[float] = []
first_solved = None

for epi in range(num_episodes):
    state, _ = env.reset(seed=epi)
    for t in range(500):
        action = pi.act(state)
        state, reward, terminated, truncated, _ = env.step(action)
        pi.rewards.append(reward)
        if terminated or truncated:
            break
    loss = train_episode(pi, optimizer)
    total_reward = float(sum(pi.rewards))
    episode_returns.append(total_reward)
    if first_solved is None and total_reward > 475.0:
        first_solved = epi
    pi.onpolicy_reset()

    if _use_wandb:
        window = episode_returns[-50:]
        wandb.log({
            "episode": epi,
            "loss": loss,
            "return": total_reward,
            "return_mean_50": float(np.mean(window)),
        })

env.close()

mean_last_50 = float(np.mean(episode_returns[-50:]))
peak_return = float(max(episode_returns))
print(f"Episodes run: {num_episodes}")
print(f"First episode with return > 475: {first_solved}")
print(f"Peak return: {peak_return:.0f}")
print(f"Mean return over last 50 episodes: {mean_last_50:.1f}")

if _use_wandb:
    wandb.summary["first_solved_episode"] = first_solved
    wandb.summary["peak_return"] = peak_return
    wandb.summary["mean_return_last_50"] = mean_last_50

## Learning curve

Per-episode returns (thin) and a 25-episode moving average (bold) show how quickly the policy climbs toward the 500 cap.

In [ ]:
returns = np.array(episode_returns, dtype=np.float32)
window = 25
if len(returns) >= window:
    smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")
else:
    smoothed = returns

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(returns, alpha=0.35, color="#2563eb", label="episode return")
if len(smoothed) > 0:
    ax.plot(np.arange(len(smoothed)) + window - 1, smoothed, color="#0f172a", linewidth=2.0, label=f"{window}-episode moving avg")
ax.axhline(475, color="#059669", linestyle="--", linewidth=1, label="solved threshold (475)")
ax.set_xlabel("episode")
ax.set_ylabel("total return")
ax.set_title("REINFORCE on CartPole-v1 — learning curve")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "learning-curve.svg", format="svg", bbox_inches="tight")
plt.close(fig)

![REINFORCE learning curve on CartPole-v1 — per-episode returns and a 25-episode moving average rising toward the 475 solved threshold](images/learning-curve.svg)

*Learning curve — per-episode return and moving average. The curve is not monotonic; see the [parent lecture page](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce#why-the-learning-curve-is-not-monotonic) for why, and the variance-reduction discussion that follows.*